In [1]:
import numpy as np
import pandas as pd
import networkx as nx
import torch
import copy
import itertools

from pymatgen.core.structure import Structure
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.graphs import StructureGraph
from pymatgen.analysis import local_env

from networkx.algorithms.components import is_connected

from sklearn.metrics import accuracy_score, recall_score, precision_score

from torch_scatter import scatter

from p_tqdm import p_umap

import ast
#import the random function library
import random

import os 

from tqdm.auto import tqdm
tqdm.pandas()

CrystalNN = local_env.CrystalNN(
    distance_cutoffs=None, x_diff_weight=-1, porous_adjustment=False)

from data_utils import * 

import sys

# #read in the worker number 
# worker_num = int(sys.argv[1])

data_dir = '/home/gridsan/tmackey/cdvae/data/mp_20/'

#load in the data 
train_df = pd.read_csv(data_dir + 'train.csv')
test_df = pd.read_csv(data_dir + 'test.csv')
val_df = pd.read_csv(data_dir + 'val.csv')


data_frames = {"train": train_df, "test": test_df, "val": val_df}

for name, df in data_frames.items():

    #assuming you want 20 workers 
    num_splits = 20
    chunk_size = np.ceil(len(df)/num_splits)

    worker_num = 0 #change when move to python script 
    
    start_index = int(worker_num*chunk_size)
    end_index = int(min(start_index + chunk_size, len(df))) #prevents end index > len(df)
    sub_df = df.iloc[start_index:end_index].copy()
    sub_crystals = sub_df['cif'].progress_apply(build_crystal)
    sub_graphs = sub_crystals.progress_apply(build_crystal_graph)

    materials_ids = sub_df['material_id'].values

    #make a dictionary using the materials_ids as keys and the graphs as values
    graph_dict = dict(zip(materials_ids, sub_graphs))

    #save the dictionary to a file
    torch.save(graph_dict, data_dir + 'graph_preloading_data/{}_{}.pt'.format(name, worker_num))
    
    print('Saved {}_{}.pt'.format(name, worker_num))

After the data is generated, all we have to do is merge it back together

In [16]:
import numpy as np
import torch 

In [17]:
#assuming all of the data was stored as .pt dictionaries of string keys and graph values
num_workers = 20 
dataset_names =  ['train', 'test', 'val']
for name in dataset_names:
    total_dict = {}
    for worker_num in range(num_workers):
        chunk = torch.load('/home/gridsan/tmackey/cdvae/data/mp_20/graph_preloading_data/{}_{}.pt'.format(name, worker_num))
        total_dict.update(chunk)
    torch.save(total_dict, '/home/gridsan/tmackey/cdvae/data/mp_20/graph_preloading_data/{}.pt'.format(name))
    print('Saved {}.pt'.format(name))

Saved train.pt
Saved test.pt
Saved val.pt


In [42]:
# num_workers = 20 

# for name in ['train', 'test', 'val']:
#     print(name)
#     #load in the data 
#     data_frames = []
#     for worker_num in range(num_workers):
#         data_frames.append(pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/graph_preloading_data/{}_{}.csv'.format(name, worker_num)))
#     df = pd.concat(data_frames)
#     df.to_csv('/home/gridsan/tmackey/cdvae/data/mp_20/graph_preloading_data/{}.csv'.format(name))
#     print('Saved {}.csv'.format(name))

train
Saved train.csv
test
Saved test.csv
val
Saved val.csv


Now we just need to double check that the data was made correctly

In [18]:
import numpy as np
import torch
#load in the new train, test, and val dictionaries

train_dict = torch.load('/home/gridsan/tmackey/cdvae/data/mp_20/graph_preloading_data/train.pt')
test_dict = torch.load('/home/gridsan/tmackey/cdvae/data/mp_20/graph_preloading_data/test.pt')
val_dict = torch.load('/home/gridsan/tmackey/cdvae/data/mp_20/graph_preloading_data/val.pt')

train_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/train.csv')
test_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/test.csv')
val_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/val.csv')

In [19]:
#see if the data is ordered correctly
train_list = list(train_dict.values())
test_list = list(test_dict.values())
val_list = list(val_dict.values())

In [20]:
#get a random sample of 10 indices from the train_df
indices = random.sample(range(0, len(train_df)), 10)
for index in indices: 
    #regenerate the graph for the first entry in train_df
    first_entry_check = build_crystal_graph(build_crystal(train_df['cif'][index]))
    #get the materials id
    material_id = train_df['material_id'].iloc[index]
    for sub_index in range(7):
        print(np.mean(first_entry_check[sub_index] == train_dict[material_id][sub_index]))
    #print(first_entry_check == train_dict[material_id])

1.0
1.0
1.0
1.0
1.0
1.0
1.0


/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))


1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0


In [94]:
#copy the train.pt, test.pt and val.pt files out of the graph_preloading_data folder and into the main data folder
import shutil
import os

data_source = "mp_20"

for name in ['train', 'test', 'val']:
    shutil.copy('/home/gridsan/tmackey/cdvae/data/{}/graph_preloading_data/{}.pt'.format(data_source, name), '/home/gridsan/tmackey/cdvae/data/{}/{}.pt'.format(data_source, name))


Repeat the process for the perov data


In [6]:
import numpy as np
import torch 

#assuming all of the data was stored as .pt dictionaries of string keys and graph values
num_workers = 20 
dataset_names =  ['train', 'test', 'val']

data_source = "perov_5"

for name in dataset_names:
    total_dict = {}
    for worker_num in range(num_workers):
        chunk = torch.load('/home/gridsan/tmackey/cdvae/data/{}/graph_preloading_data/{}_{}.pt'.format(data_source, name, worker_num))
        total_dict.update(chunk)
    torch.save(total_dict, '/home/gridsan/tmackey/cdvae/data/{}/graph_preloading_data/{}.pt'.format(data_source, name))
    print('Saved {}.pt'.format(name))

Saved train.pt
Saved test.pt
Saved val.pt


In [14]:
import numpy as np
import torch
#load in the new train, test, and val dictionaries

data_source = "perov_5"

train_dict = torch.load('/home/gridsan/tmackey/cdvae/data/{}/graph_preloading_data/train.pt'.format(data_source))
test_dict = torch.load('/home/gridsan/tmackey/cdvae/data/{}/graph_preloading_data/test.pt'.format(data_source))
val_dict = torch.load('/home/gridsan/tmackey/cdvae/data/{}/graph_preloading_data/val.pt'.format(data_source))

train_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/{}/train.csv'.format(data_source))
test_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/{}/test.csv'.format(data_source))
val_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/{}/val.csv'.format(data_source))
                      
#get a random sample of 10 indices from the train_df
indices = random.sample(range(0, len(train_df)), 10)
for index in indices: 
    #regenerate the graph for the first entry in train_df
    first_entry_check = build_crystal_graph(build_crystal(train_df['cif'][index]))
    #get the materials id
    material_id = train_df['material_id'].iloc[index]
    for sub_index in range(7):
        print(np.mean(first_entry_check[sub_index] == train_dict[material_id][sub_index]))
    #print(first_entry_check == train_dict[material_id])

1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0


In [15]:
#copy the train.pt, test.pt and val.pt files out of the graph_preloading_data folder and into the main data folder
import shutil
import os

data_source = "perov_5"

for name in ['train', 'test', 'val']:
    shutil.copy('/home/gridsan/tmackey/cdvae/data/{}/graph_preloading_data/{}.pt'.format(data_source, name), '/home/gridsan/tmackey/cdvae/data/{}/{}.pt'.format(data_source, name))
